# Checkpointing: Truncating Lineage and Durable State Preservation

**Module**: 11_fault_tolerance_and_reliability  
**Notebook**: 04_checkpointing.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How checkpointing saves materialized data to durable storage and completely severs the upstream lineage graph.
- **Why does this topic exist?**: Recursive or iterative algorithms generate massive DAGs that cause JVM StackOverflowError unless checkpointed.
- **What problem does it solve?**: JVM crashing due to 500-step iterative DAG lineage.
- **Where is it used in real-world Data Engineering?**: Iterative graph algorithms, ML training, and streaming recovery.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Advanced |
| **Estimated effort** | 3–4 h |
| **Prerequisites** | Lineage, persistence, reliable storage |
| **Required depth** | Deep working knowledge: can choose between cache, checkpoint, and table materialization |
| **Data Engineer** | Use checkpointing to truncate lineage and bound recovery cost. |
| **Senior Data Engineer** | Choose checkpoint locations and lifecycle policies. |
| **Spark specialist** | Reason about eager versus lazy checkpointing and reliable storage requirements. |
| **Next topic** | `11_fault_tolerance_and_reliability/05_fault_tolerance_experiments.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Checkpointing writes an intermediate dataset to reliable storage and cuts the lineage. It is different from
caching: a cache keeps lineage and can be evicted, while a checkpoint is meant to survive and to make recovery
independent of earlier stages. The cost is I/O and storage; the benefit is bounded recovery for long or iterative
pipelines.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Checkpoint** | Durable materialization | Writes to a reliable filesystem and truncates lineage |
| **Local checkpoint** | Executor-local materialization | Cuts lineage but is not durable |
| **Checkpoint directory** | Storage location | Must be configured before use |
| **Eager checkpoint** | Materializes immediately | Dataset checkpoint triggers computation |
| **Lazy checkpoint** | Materializes on action | RDD checkpoint waits for a job |
| **Cache vs checkpoint** | Different purposes | Cache speeds reuse; checkpoint bounds recovery |

### How it works, step by step

1. The dataset is marked for checkpointing.
2. On materialization, partitions are written to the checkpoint directory.
3. The resulting dataset reads from the checkpoint instead of the original lineage.
4. Later failures recover from the checkpoint rather than recomputing earlier stages.
5. The checkpoint remains until it is deleted or the job/session cleanup policy removes it.
6. Local checkpoints use executor storage and do not survive executor loss.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Checkpoint vs Persist
- `persist()`: Keeps data in memory/disk; **retains full lineage**.
- `checkpoint()`: Saves to reliable storage; **truncates lineage completely**.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("Checkpointing_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
chk_dir = "/opt/spark-data/warehouse/rdd_checkpoint_dir"
os.system(f"rm -rf {chk_dir}")
spark.sparkContext.setCheckpointDir(f"file://{chk_dir}")

# Build RDD and checkpoint
rdd = spark.sparkContext.parallelize(range(20), 2).map(lambda x: x * 3)
rdd.checkpoint()

print("Lineage before Action:")
print(rdd.toDebugString().decode('utf-8'))

# Action triggers materialization and lineage truncation
rdd.count()

print("\nLineage AFTER Action (Notice: Lineage is TRUNCATED!):")
print(rdd.toDebugString().decode('utf-8'))

---

## Experiment: Verifying Lineage Truncation

- **1. What are we testing?**: Verify that toDebugString() points directly to CheckpointRDD after action.
- **2. What do we expect?**: Original parallelize and map steps are completely severed.

In [ ]:
print('Is RDD checkpointed?:', rdd.isCheckpointed())

**3. What happened?**: isCheckpointed returned True; parent steps vanished from debug string.  
**4. Why did it happen?**: Spark wrote the materialized partition files to disk and reset the parent pointer.  
**5. Practical Engineering Lesson**: Checkpointing cuts the lineage graph, freeing JVM memory from deep recursion.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Iterative model training

- **Industry**: Machine learning
- **Scenario**: A model trains for many iterations and the lineage grows with each iteration.
- **Problem**: A failure late in training recomputes the entire chain and can overflow the driver.
- **Why this concept is relevant**: Checkpointing the feature dataset or periodic model state bounds the lineage and recovery cost.
- **Implementation idea**: Checkpoint the stable feature stage and persist model state at intervals.
- **Expected result**: Faster recovery and bounded lineage growth.
- **Alternative**: Write features to a feature store.
- **Trade-offs**: A feature store supports reuse across jobs but adds governance and freshness work.

### Use Case 2: Regulatory audit trail

- **Industry**: Financial services
- **Scenario**: A risk pipeline must be reproducible and recoverable for audit.
- **Problem**: Recomputing from raw data may produce different results if source data changed.
- **Why this concept is relevant**: Checkpointing key intermediate results preserves the exact inputs used for a run.
- **Implementation idea**: Write audited intermediates to immutable storage with run metadata.
- **Expected result**: A reproducible audit trail.
- **Alternative**: Store only the final output and source snapshots.
- **Trade-offs**: Final-output storage is smaller but may not support detailed investigation.

---

## Business Value

- Bounded recovery time for long pipelines.
- Lower risk of driver lineage overflow.
- Reproducible intermediate state for audits.
- Clear separation between reuse (cache) and durability (checkpoint).

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Configure a reliable checkpoint directory before use.
- Use checkpoints for long lineages, not for every step.
- Clean up checkpoints with a lifecycle policy.
- Do not confuse local checkpoints with durable checkpoints.
- Verify that the checkpoint is actually read after a failure.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Compare cache and checkpoint

**Steps**

1. Build a long lineage and cache an intermediate DataFrame.
2. Run a job, then inspect the Storage tab and the plan.
3. Checkpoint the same DataFrame and inspect the new plan and checkpoint directory.

**What to observe**: The cache keeps lineage and can be evicted; the checkpoint reads from durable storage and truncates lineage.

**Healthy result**: The checkpoint is used after a simulated failure.

**Problematic result**: The checkpoint directory is missing or the dataset is recomputed from source.

**Engineering takeaway**: Checkpoint for recovery; cache for reuse.

---

## Trade-offs and Alternatives

### Advantages

- Truncates lineage
- Bounded recovery
- Durable intermediate state

### Limitations

- Adds I/O and storage
- Checkpoint management required
- Not a substitute for idempotent outputs

### When to use it

- Iterative algorithms
- Long ETL
- Audit/reproducibility

### When not to use it

- Checkpointing every step
- Using local checkpoints for durability
- Leaving orphaned checkpoints

### Alternatives inside Spark

- RDD checkpoint
- Dataset checkpoint
- Local checkpoint

### Alternatives outside Spark

- Feature stores
- Materialized tables
- Workflow artifact storage

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Set Checkpoint Dir
**Problem Statement**: Set checkpoint directory on SparkContext.  
**Input / Context**: setCheckpointDir

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): DataFrame Checkpoint
**Problem Statement**: Checkpoint a DataFrame using df.checkpoint().  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Eager vs Non-Eager Checkpoint
**Problem Statement**: Explain df.checkpoint(eager=False).  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The checkpoint that was never used

**Scenario**: A team checkpoints a DataFrame, but recovery still recomputes the full pipeline.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the checkpoint completed and the directory contains data.
2. Verify that the downstream job reads the checkpointed DataFrame, not the original.
3. Confirm the checkpoint directory is accessible after restart.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Set Checkpoint Dir):
```python
spark.sparkContext.setCheckpointDir('file:///opt/spark-data/warehouse/chk')
```

#### Solution 2 (DataFrame Checkpoint):
```python
df = spark.range(10); df_c = df.checkpoint(); print('DF checkpointed:', df_c.isLocal)
```

#### Solution 3 (Eager vs Non-Eager Checkpoint):
```python
print('eager=True executes immediately; eager=False waits for downstream action.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Lineage Cut**: Checkpointing permanently severs parent dependencies.
- **Durable Storage**: Saved to reliable disk/HDFS.

### Essential Technical Terms
- `CheckpointRDD`: RDD reading directly from checkpoint files.
- `setCheckpointDir`: Registers checkpoint storage path.

### Common Engineering Mistakes
- **Using persist instead of checkpoint for deep loops**: Causes JVM StackOverflowError.

### Interview Questions on This Topic
- **Q: What is the difference between persist() and checkpoint()?**
  - *A*: persist() keeps data in executor cache while retaining the lineage graph; checkpoint() writes data to durable storage and permanently severs the lineage graph.


### More Interview Questions

- **Q: What is the difference between cache and checkpoint?**
  - *A*: A cache stores data for reuse but keeps lineage and can be evicted; a checkpoint writes to reliable storage and truncates lineage, making recovery independent of earlier stages.
- **Q: When should you checkpoint?**
  - *A*: For long lineages, iterative algorithms, or expensive stages where recomputation after a failure would be costly; not for every intermediate step.

### What to Study Next
Open 05_fault_tolerance_experiments.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')